In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:
from transformers import Qwen2_5OmniProcessor, AutoProcessor

# model_name_or_path = "/projects/bhuang/models/llm/pretrained/Qwen/Qwen2.5-Omni-3B"
# model_name_or_path = "/home/bhuang/llm/trl/outputs/intent_classification/audio_ft/sga/sft_qwen2_5_omni_3b_lora_r64_ep5_bs64_lr2e4_merged"
model_name_or_path = "/home/bhuang/llm/trl/outputs/intent_classification/audio_ft/sga/sft_qwen2_5_omni_3b_lora_r64_ep10_bs64_lr2e4_merged"

processor = Qwen2_5OmniProcessor.from_pretrained(model_name_or_path)

type(processor)

In [ ]:
import torch
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniThinkerForConditionalGeneration

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    model_name_or_path,
    dtype=torch.bfloat16,
    device_map="cuda",
).eval()

# model.disable_talker()

type(model)

In [ ]:
# from qwen_omni_utils import process_mm_info

# audio_filepath = "/home/bhuang/llm/momo/intent_classification/data/sga/audio_degraded/000000.wav"
audio_filepath = "/home/bhuang/llm/momo/intent_classification/data/databank/audio_degraded/train/000000.wav"

messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": "You are an assistant that transcribes speech accurately.",
            }
        ],
    },
    {
        "role": "user",
        "content": [
            # voxtral
            # {"type": "audio", "url": audio_url},
            # {"type": "audio", "path": audio_filepath},
            # {"type": "audio", "base64": audio_base64},
            # gemma-3n, qwen-omni
            {"type": "audio", "audio": audio_filepath},
            {"type": "text", "text": "Please transcribe this audio."},
            # {"type": "text", "text": "Please recognize the intent of this audio."},
        ],
    },
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    # train
    # add_generation_prompt=False,
    # padding=True,
)

# # set use audio in video
# USE_AUDIO_IN_VIDEO = True
# # Preparation for inference
# text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
# audios, images, videos = process_mm_info(messages, use_audio_in_video=USE_AUDIO_IN_VIDEO)
# inputs = processor(text=text, audio=audios, images=images, videos=videos, return_tensors="pt", padding=True, use_audio_in_video=USE_AUDIO_IN_VIDEO)

list(inputs.keys()), inputs

In [ ]:
with torch.inference_mode():
    outputs = model.generate(
        **inputs.to(model.device, dtype=model.dtype),
        max_new_tokens=100,
        # do_sample=False,
        # disable_compile=True,
    )

outputs = outputs[0]

decoded_outputs = processor.batch_decode(
    outputs[inputs["input_ids"].shape[1]:],  # todo: why single dimension?
    skip_special_tokens=True,
    # clean_up_tokenization_spaces=True,
)

print(decoded_outputs[0])